# Vision Transformer dari Nol pada CIFAR-10

**Buku:** *Deep Learning dengan Python* — Bab 10, Vision Transformer dan Arsitektur Visi Modern

Gambar dipotong menjadi *patch* 4×4, setiap *patch* diproyeksikan menjadi token, lalu diproses oleh tumpukan blok Transformer (*pre-norm*).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Kode bersifat *backend-agnostik* (TensorFlow, JAX, atau PyTorch).

**Perkiraan waktu (GPU T4):** ±25 menit untuk 40 epoch

**Isi notebook:**
- Visualisasi pemotongan gambar menjadi *patch*.
- Lapisan `EmbeddingPatch`: proyeksi linear (konvolusi bertitik langkah = ukuran *patch*) + embedding posisi.
- Blok ViT *pre-norm*, optimizer AdamW, dan augmentasi.
- Melihat struktur 2D yang dipelajari embedding posisi.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Data dan *patch*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras import layers, ops

(x_latih, y_latih), (x_uji, y_uji) = keras.datasets.cifar10.load_data()
x_latih, x_val = x_latih[5000:], x_latih[:5000]
y_latih, y_val = y_latih[5000:], y_latih[:5000]

UKURAN, PATCH = 32, 4
N_PATCH = (UKURAN // PATCH) ** 2               # 64 token per gambar

g = x_latih[0]
patch = g.reshape(8, PATCH, 8, PATCH, 3).transpose(0, 2, 1, 3, 4)
fig = plt.figure(figsize=(9, 4.5))
ax = fig.add_subplot(1, 2, 1); ax.imshow(g); ax.set_title("Gambar 32×32"); ax.axis("off")
for i in range(8):
    for j in range(8):
        ax = fig.add_subplot(8, 16, i * 16 + j + 9); ax.imshow(patch[i, j]); ax.axis("off")
plt.suptitle("64 patch berukuran 4×4 = 64 token"); plt.show()

## 2. Komponen ViT

In [ ]:
class EmbeddingPatch(layers.Layer):
    """Patch -> vektor (proyeksi linear) + embedding posisi yang dipelajari."""
    def __init__(self, n_patch, dim, ukuran_patch, **kwargs):
        super().__init__(**kwargs)
        # Conv2D dengan kernel = stride = ukuran patch identik dengan proyeksi linear tiap patch
        self.proyeksi = layers.Conv2D(dim, ukuran_patch, strides=ukuran_patch)
        self.ratakan = layers.Reshape((n_patch, dim))
        self.posisi = layers.Embedding(n_patch, dim)
        self.n_patch = n_patch
        self.cfg = dict(n_patch=n_patch, dim=dim, ukuran_patch=ukuran_patch)

    def call(self, x):
        return self.ratakan(self.proyeksi(x)) + self.posisi(ops.arange(self.n_patch))

    def get_config(self):
        return {**super().get_config(), **self.cfg}


def blok_vit(x, dim, n_head, dim_ff, dropout=0.1):
    h = layers.LayerNormalization(epsilon=1e-6)(x)                        # pre-norm
    h = layers.MultiHeadAttention(num_heads=n_head, key_dim=dim // n_head, dropout=dropout)(h, h)
    x = layers.add([x, h])
    h = layers.LayerNormalization(epsilon=1e-6)(x)
    h = layers.Dense(dim_ff, activation="gelu")(h)
    h = layers.Dropout(dropout)(h)
    h = layers.Dense(dim)(h)
    return layers.add([x, h])

In [ ]:
DIM, N_HEAD, DIM_FF, N_BLOK = 128, 4, 256, 6

normalisasi = layers.Normalization()
normalisasi.adapt(x_latih.astype("float32"))

augmentasi = keras.Sequential([layers.RandomFlip("horizontal"),
                               layers.RandomTranslation(0.1, 0.1),
                               layers.RandomZoom(0.1)], name="augmentasi")

masukan = keras.Input(shape=(UKURAN, UKURAN, 3))
x = normalisasi(masukan)
x = augmentasi(x)
x = EmbeddingPatch(N_PATCH, DIM, PATCH, name="embedding_patch")(x)
for _ in range(N_BLOK):
    x = blok_vit(x, DIM, N_HEAD, DIM_FF)
x = layers.LayerNormalization(epsilon=1e-6)(x)
x = layers.GlobalAveragePooling1D()(x)          # alternatif token [CLS] pada ViT asli
x = layers.Dropout(0.3)(x)
vit = keras.Model(masukan, layers.Dense(10)(x))  # logit
vit.summary()

## 3. Pelatihan

In [ ]:
vit.compile(optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"])
riwayat = vit.fit(x_latih, y_latih, batch_size=128, epochs=40, validation_data=(x_val, y_val),
                  callbacks=[keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)])
print(f"ViT, akurasi uji: {vit.evaluate(x_uji, y_uji, verbose=0)[1]:.2%}")

plt.plot(riwayat.history["accuracy"], label="Latih")
plt.plot(riwayat.history["val_accuracy"], label="Validasi")
plt.xlabel("Epoch"); plt.ylabel("Akurasi"); plt.legend(); plt.show()

Bandingkan dengan convnet residual di `bab07_02` yang memiliki jumlah parameter serupa. Pada data kecil seperti CIFAR-10 (45 ribu gambar), ViT dari nol biasanya kalah dari convnet karena tidak punya bias induktif lokalitas. Keunggulan ViT baru muncul dengan pretraining pada data sangat besar (lihat `bab10_02`).

## 4. Apa yang dipelajari embedding posisi?
Untuk setiap *patch*, kita gambar kemiripan kosinus embedding posisinya dengan semua *patch* lain. Tanpa diberi tahu, model menemukan struktur baris dan kolom 2D.

In [ ]:
pos = keras.ops.convert_to_numpy(vit.get_layer("embedding_patch").posisi.embeddings)
pos = pos / np.linalg.norm(pos, axis=1, keepdims=True)
kemiripan = pos @ pos.T

fig, ax = plt.subplots(8, 8, figsize=(8, 8))
for i in range(N_PATCH):
    ax[i // 8, i % 8].imshow(kemiripan[i].reshape(8, 8), cmap="viridis", vmin=-1, vmax=1)
    ax[i // 8, i % 8].axis("off")
plt.suptitle("Kemiripan kosinus embedding posisi"); plt.show()